# Analisis Spasial & Kelayakan Buka Kafe di Sekitar Kampus Surabaya

**Mata Kuliah:** Computer Graphics (AFL 2) — Program Studi Informatika, Universitas Ciputra Surabaya (T.A. 2026/2027)  
**Dokumen Proposal Acuan:** `Proposal AFL 2 Computer Graphics.pdf`  
**Tim Pengembang:**
- Filemon Jose Hagen (0706012410016)
- Hafizd Dharma Saputra Hardiono (0706012410023)
- Flabianos Jason Muljono (0706012410019)

---

## 1. Pendahuluan & Koordinasi Kontrak Backend
Notebook ini dirancang untuk menjalankan analisis geospasial luring (*offline analysis*) berbasis pustaka GIS Python bawaan proyek, mengoordinasikan kontrak backend dengan modul perencana (*cafe planner*), memvalidasi **16 kandidat lahan observasi nyata Google Street View (PR#2)**, mengekspor data observasi valid ke format CSV, dan menyusun ringkasan transparan (*summary report*).

### Kontrak Geospasial & Logika Backend:
1. **Sistem Proyeksi:** Transformasi koordinat WGS84 (`EPSG:4326`) ke UTM Zone 49S (`EPSG:32749`) yang bersatuan meter untuk akurasi metrik tinggi di wilayah Surabaya.
2. **Buffer Spasial:** Pembentukan zona penyangga 1.000 meter (1 km) secara metrik dari 8 titik kampus sasaran (Unair Kampus A, B, C; UKWMS Dinoyo, Kalijudan, Pakuwon City; Ubaya Ngagel, Tenggilis).
3. **Taksonomi 5 Status Lahan:**
   - `Lahan kosong` (*Eligible / Memenuhi Syarat*)
   - `Lahan yang tersedia untuk disewakan` (*Eligible / Memenuhi Syarat*)
   - `Rumah atau bangunan tempat tinggal` (*Ineligible / Bukan Peruntukan*)
   - `Bangunan yang telah digunakan untuk fungsi lain` (*Ineligible / Terpakai*)
   - `Lahan yang statusnya belum terverifikasi` (*Ineligible / Default Fallback*)
4. **Standar Anti-Fabrikasi:** Hanya data yang benar-benar diobservasi melalui citra Street View yang dicatat. Luas tanah dan harga sewa diisi `None` (kosong) karena citra visual tidak menampilkan nilai transaksi sewa yang pasti.

In [1]:
import os
import sys
import json
import csv
import math
from pathlib import Path

# Cari root direktori repositori secara fleksibel
cwd = Path.cwd().resolve()
if (cwd / "scripts" / "gis_pipeline.py").exists():
    REPO_ROOT = cwd
elif (cwd.parent / "scripts" / "gis_pipeline.py").exists():
    REPO_ROOT = cwd.parent
else:
    REPO_ROOT = Path("/Users/josehagen/Documents/GitHub/Comgraph-AFL-2").resolve()

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Impor pustaka GIS pipeline dan konstanta kontrak
from scripts.gis_pipeline import (
    GISPipeline,
    VALID_LAND_STATUSES,
    ELIGIBLE_STATUSES,
    RENTABLE_OR_SALE_STATUSES,
    NON_RENTABLE_OR_SALE_STATUSES,
    CAMPUS_POLYGONS_WGS84
)

print(f"✓ Repositori root: {REPO_ROOT}")
print(f"✓ Modul scripts.gis_pipeline berhasil diimpor!")
print(f"✓ Status lahan valid: {VALID_LAND_STATUSES}")
print(f"✓ Status memenuhi syarat (eligible): {ELIGIBLE_STATUSES}")

✓ Repositori root: /Users/josehagen/Documents/GitHub/Comgraph-AFL-2
✓ Modul scripts.gis_pipeline berhasil diimpor!
✓ Status lahan valid: ['Lahan kosong', 'Lahan yang tersedia untuk disewakan', 'Rumah atau bangunan tempat tinggal', 'Bangunan yang telah digunakan untuk fungsi lain', 'Lahan yang statusnya belum terverifikasi']
✓ Status memenuhi syarat (eligible): {'Lahan yang tersedia untuk disewakan', 'Lahan kosong'}


## 2. Inisialisasi & Eksekusi Pipeline GIS
Pada tahap ini, kita mengeksekusi pipeline GIS terhadap berkas masukan mentah (`data/raw/universities.geojson` dan `data/raw/survey_land_candidates.geojson`). Pipeline akan:
1. Memvalidasi 8 koordinat kampus dan poligon perimeter OSM.
2. Membangkitkan buffer radial 1 km dan buffer poligon terproyeksi.
3. Melakukan uji spasial *point-to-polygon* dan *point-in-buffer* terhadap seluruh kandidat.
4. Memetakan konteks 206 kafe OpenStreetMap di Surabaya.

In [2]:
# Inisialisasi objek GISPipeline
pipeline = GISPipeline()

# Tentukan path file input dan output
uni_path = REPO_ROOT / "data" / "raw" / "universities.geojson"
candidates_path = REPO_ROOT / "data" / "raw" / "survey_land_candidates.geojson"
output_dir = REPO_ROOT / "data" / "processed"

# Jalankan pipeline pemrosesan data
pipeline_result = pipeline.process(
    universities_path=str(uni_path),
    candidates_path=str(candidates_path),
    output_dir=str(output_dir)
)

print("=== HASIL EKSEKUSI PIPELINE GIS ===")
print(f"Total Kampus Sasaran       : {pipeline_result['universities_count']} kampus")
print(f"Total Buffer Radial (1 km) : {pipeline_result['buffers_count']} geometri")
print(f"Total Buffer Poligon (1 km): {pipeline_result['polygon_buffers_count']} geometri")
print(f"Total Kandidat Lahan       : {pipeline_result['candidates_count']} observasi Street View")
print(f"Total Kafe OSM di Surabaya : {pipeline_result['cafes_count']} titik")
print(f"Kafe OSM dalam Buffer 1 km : {pipeline_result['cafes_in_buffer_count']} titik")
print("\nStatistik Kelayakan Kandidat Lahan:")
stats = pipeline_result['stats']
print(f"  - Total Observasi        : {stats['total_candidates']}")
print(f"  - Dalam Buffer (<= 1 km) : {stats['in_buffer_count']}")
print(f"  - Luar Buffer (> 1 km)   : {stats['out_of_buffer_count']}")
print(f"  - Memenuhi Syarat        : {stats['eligible_count']} ({stats['eligible_count']/stats['total_candidates']*100:.1f}%)")
print(f"  - Tidak Memenuhi Syarat  : {stats['ineligible_count']} ({stats['ineligible_count']/stats['total_candidates']*100:.1f}%)")
print("\nDistribusi Status Lahan:")
for status, count in stats['status_counts'].items():
    print(f"  * {status:<45}: {count} lokasi")

=== HASIL EKSEKUSI PIPELINE GIS ===
Total Kampus Sasaran       : 8 kampus
Total Buffer Radial (1 km) : 8 geometri
Total Buffer Poligon (1 km): 8 geometri
Total Kandidat Lahan       : 16 observasi Street View
Total Kafe OSM di Surabaya : 206 titik
Kafe OSM dalam Buffer 1 km : 59 titik

Statistik Kelayakan Kandidat Lahan:
  - Total Observasi        : 16
  - Dalam Buffer (<= 1 km) : 16
  - Luar Buffer (> 1 km)   : 0
  - Memenuhi Syarat        : 4 (25.0%)
  - Tidak Memenuhi Syarat  : 12 (75.0%)

Distribusi Status Lahan:
  * Lahan kosong                                 : 3 lokasi
  * Lahan yang tersedia untuk disewakan          : 1 lokasi
  * Rumah atau bangunan tempat tinggal           : 5 lokasi
  * Bangunan yang telah digunakan untuk fungsi lain: 6 lokasi
  * Lahan yang statusnya belum terverifikasi     : 1 lokasi


## 3. Audit Rinci 16 Observasi Nyata Google Street View (PR#2)
Seluruh 16 observasi kandidat lahan berasal dari pengamatan visual Google Street View nyata dengan tautan panorama kanonikal (`pano_id`), arah pandang kamera (`heading_deg`), koordinat lintang/bujur berpresisi penuh, serta tanggal pengambilan gambar (`imagery_date`).

Berikut adalah tabel audit menyeluruh untuk memverifikasi lokasi dan atribut setiap kandidat.

In [3]:
processed_candidates_file = output_dir / "land_candidates.geojson"
with open(processed_candidates_file, "r", encoding="utf-8") as f:
    land_geojson = json.load(f)

features = land_geojson.get("features", [])

print(f"{'No':<3} | {'ID':<22} | {'Kampus Sasaran':<16} | {'Jarak':<8} | {'Status Lahan':<35} | {'Eligible':<8} | {'Pano ID'}")
print("-" * 125)
for idx, feat in enumerate(features, 1):
    p = feat["properties"]
    print(f"{idx:<3} | {p['id']:<22} | {p['nearest_university_id']:<16} | {p['distance_meters']:>6.1f}m | {p['status']:<35} | {str(p['is_eligible']):<8} | {p['pano_id']}")
print("-" * 125)

No  | ID                     | Kampus Sasaran   | Jarak    | Status Lahan                        | Eligible | Pano ID
-----------------------------------------------------------------------------------------------------------------------------
1   | SV-UNAIR-A-01          | UNAIR-KAMPUS-A   |  370.8m | Rumah atau bangunan tempat tinggal  | False    | mwLkdUHagJ_QqnU2evtIKg
2   | SV-UNAIR-A-02          | UNAIR-KAMPUS-B   |  426.9m | Bangunan yang telah digunakan untuk fungsi lain | False    | QLLedWYGWLsnGHUE5Ao3Mg
3   | SV-UNAIR-A-03          | UNAIR-KAMPUS-A   |  178.2m | Bangunan yang telah digunakan untuk fungsi lain | False    | cO08ZfzK6yim8eqXhWTMGA
4   | SV-UNAIR-C-01          | UNAIR-KAMPUS-C   |  541.0m | Rumah atau bangunan tempat tinggal  | False    | GGwyMtSGD2Yt3CYr1sF4oQ
5   | SV-UNAIR-C-02          | UNAIR-KAMPUS-C   |  465.8m | Lahan yang tersedia untuk disewakan | True     | axuDfMvoxSFQUquRPNTw9A
6   | SV-UKWMS-PAKUWON-01    | UKWMS-PAKUWON    |  258.6m | Lahan kosong

## 4. Analisis Mendalam 4 Kandidat Lahan yang Memenuhi Syarat (*Eligible*)
Berdasarkan aturan seleksi dua tahap:
1. **Syarat Spasial:** Berada dalam buffer $\le 1.000	ext{ meter}$ dari kampus sasaran.
2. **Syarat Status:** Berstatus `Lahan kosong` atau `Lahan yang tersedia untuk disewakan`.

Terdapat **4 kandidat lahan** yang lolos kualifikasi dan dapat digunakan dalam modul perencana kafe (*cafe planning module*).

In [4]:
eligible_features = [f for f in features if f["properties"]["is_eligible"]]

print(f"Ditemukan {len(eligible_features)} kandidat lahan yang memenuhi syarat perencana kafe:\n")
for i, feat in enumerate(eligible_features, 1):
    p = feat["properties"]
    geom = feat["geometry"]
    print(f"[{i}] {p['id']} - {p['name']}")
    print(f"    Alamat      : {p['address']}")
    print(f"    Kampus Dekat: {p['nearest_university_name']} ({p['nearest_university_id']})")
    print(f"    Jarak       : {p['distance_meters']:.1f} m (ke titik) | {p['distance_to_polygon_meters']:.1f} m (ke perimeter poligon)")
    print(f"    Status Lahan: {p['status']}")
    print(f"    Koordinat   : Lon {geom['coordinates'][0]:.6f}, Lat {geom['coordinates'][1]:.6f}")
    print(f"    Tanggal Foto: {p['imagery_date']} (Pano ID: {p['pano_id']}, Heading: {p['heading_deg']}°)")
    print(f"    Catatan Tim : {p['notes']}")
    print(f"    Tautan Bukti: {p['evidence_url']}")
    print()

Ditemukan 4 kandidat lahan yang memenuhi syarat perencana kafe:

[1] SV-UNAIR-C-02 - Lahan berpagar disewakan Jl. Mulyorejo Utara 221
    Alamat      : Jl. Mulyorejo Utara No. 221, Surabaya
    Kampus Dekat: Kampus C Universitas Airlangga (UNAIR-KAMPUS-C)
    Jarak       : 465.8 m (ke titik) | 95.0 m (ke perimeter poligon)
    Status Lahan: Lahan yang tersedia untuk disewakan
    Koordinat   : Lon 112.786111, Lat -7.262087
    Tanggal Foto: 2024-06 (Pano ID: axuDfMvoxSFQUquRPNTw9A, Heading: 0°)
    Catatan Tim : Pagar bertuliskan 'Jl. Mulyorejo Utara 221' dan papan 'DISEWAKAN' dengan keterangan luas tanah (angka kurang terbaca, sekitar 1.0xx m2) dan daya listrik. Nomor kontak tidak dicatat.
    Tautan Bukti: https://www.google.com/maps/@-7.2620868,112.7861111,3a,75y,0h,90t/data=!3m6!1e1!3m4!1saxuDfMvoxSFQUquRPNTw9A!2e0!7i16384!8i8192

[2] SV-UKWMS-PAKUWON-01 - Lahan terbuka bertembok barat UKWMS Pakuwon City
    Alamat      : jalan di barat Kampus UKWMS Pakuwon City, Surabaya
    Kampu

## 5. Analisis 12 Kandidat Lahan Tidak Memenuhi Syarat (*Ineligible*)
Sebanyak 12 titik observasi diklasifikasikan sebagai *ineligible* (tidak dapat dipilih untuk perencanaan kafe) demi mencegah kesalahan peruntukan usaha pada hunian atau bangunan yang telah aktif digunakan.

In [5]:
ineligible_features = [f for f in features if not f["properties"]["is_eligible"]]

print(f"Ditemukan {len(ineligible_features)} lokasi yang TIDAK memenuhi syarat perencana kafe:\n")
by_category = {}
for feat in ineligible_features:
    p = feat["properties"]
    by_category.setdefault(p["status"], []).append(p)

for cat, items in by_category.items():
    print(f"Kategori: {cat} ({len(items)} lokasi)")
    for item in items:
        print(f"  - [{item['id']}] {item['name']} ({item['nearest_university_id']}, dist: {item['distance_meters']:.1f}m)")
        print(f"    Alasan Penolakan: {item['unavailability_reason']}")
    print()

Ditemukan 12 lokasi yang TIDAK memenuhi syarat perencana kafe:

Kategori: Rumah atau bangunan tempat tinggal (5 lokasi)
  - [SV-UNAIR-A-01] Rumah tinggal Jl. Kedung Tarukan Baru I (UNAIR-KAMPUS-A, dist: 370.8m)
    Alasan Penolakan: Status 'Rumah atau bangunan tempat tinggal' bukan peruntukan usaha kafe.
  - [SV-UNAIR-C-01] Permukiman Jl. Sutorejo Timur IV (UNAIR-KAMPUS-C, dist: 541.0m)
    Alasan Penolakan: Status 'Rumah atau bangunan tempat tinggal' bukan peruntukan usaha kafe.
  - [SV-UBAYA-TENGGILIS-03] Rumah berpagar Jl. Tenggilis Mejoyo (UBAYA-TENGGILIS, dist: 169.6m)
    Alasan Penolakan: Status 'Rumah atau bangunan tempat tinggal' bukan peruntukan usaha kafe.
  - [SV-UKWMS-DINOYO-02] Rumah berpagar Jl. Bawean (UKWMS-DINOYO, dist: 476.6m)
    Alasan Penolakan: Status 'Rumah atau bangunan tempat tinggal' bukan peruntukan usaha kafe.
  - [SV-UKWMS-KALIJUDAN-01] Rumah bertingkat Jl. Bronggalan 2 H (UKWMS-KALIJUDAN, dist: 145.7m)
    Alasan Penolakan: Status 'Rumah atau bangunan tem

## 6. Ekspor Data Kandidat Terobservasi Valid ke Format CSV
Sesuai instruksi dan panduan checklist, data kandidat lahan diekspor ke format CSV **hanya dari data observasi valid** (`data/raw/survey_land_candidates.geojson` $	o$ `data/processed/land_candidates.geojson`).

CSV ini diekspor ke:
1. `data/processed/valid_observed_candidates.csv`
2. `notebooks/valid_observed_candidates.csv`

In [6]:
csv_processed_path = output_dir / "valid_observed_candidates.csv"
csv_notebook_path = REPO_ROOT / "notebooks" / "valid_observed_candidates.csv"

# Kolom-kolom baku untuk representasi tabular
export_columns = [
    "id",
    "name",
    "address",
    "status",
    "nearest_university_id",
    "nearest_university_name",
    "distance_meters",
    "distance_to_polygon_meters",
    "is_in_buffer",
    "is_eligible",
    "is_rentable_or_sale",
    "survey_method",
    "observed_at",
    "imagery_date",
    "pano_id",
    "heading_deg",
    "location_method",
    "evidence_url",
    "notes"
]

for target_path in [csv_processed_path, csv_notebook_path]:
    target_path.parent.mkdir(parents=True, exist_ok=True)
    with open(target_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=export_columns)
        writer.writeheader()
        for feat in features:
            props = feat["properties"]
            row = {col: props.get(col) for col in export_columns}
            writer.writerow(row)

print(f"✓ Berhasil mengekspor {len(features)} baris data observasi ke:")
print(f"  1. {csv_processed_path}")
print(f"  2. {csv_notebook_path}")

# Tampilkan 3 baris pertama cuplikan CSV
print("\nCuplikan 3 baris pertama file CSV:")
with open(csv_notebook_path, "r", encoding="utf-8") as f:
    reader = csv.reader(f)
    for i, line in enumerate(reader):
        if i > 3:
            break
        print(line[:6])

✓ Berhasil mengekspor 16 baris data observasi ke:
  1. /Users/josehagen/Documents/GitHub/Comgraph-AFL-2/data/processed/valid_observed_candidates.csv
  2. /Users/josehagen/Documents/GitHub/Comgraph-AFL-2/notebooks/valid_observed_candidates.csv

Cuplikan 3 baris pertama file CSV:
['id', 'name', 'address', 'status', 'nearest_university_id', 'nearest_university_name']
['SV-UNAIR-A-01', 'Rumah tinggal Jl. Kedung Tarukan Baru I', 'Jl. Kedung Tarukan Baru I, Surabaya', 'Rumah atau bangunan tempat tinggal', 'UNAIR-KAMPUS-A', 'Kampus A Universitas Airlangga']
['SV-UNAIR-A-02', 'Lapangan olahraga berpagar Jl. Dharmawangsa', 'Jl. Dharmawangsa, Surabaya', 'Bangunan yang telah digunakan untuk fungsi lain', 'UNAIR-KAMPUS-B', 'Kampus B Universitas Airlangga']
['SV-UNAIR-A-03', 'Gedung aktif Jl. Prof. Dr. Moestopo', 'Jl. Prof. Dr. Moestopo, Surabaya', 'Bangunan yang telah digunakan untuk fungsi lain', 'UNAIR-KAMPUS-A', 'Kampus A Universitas Airlangga']


## 7. Pembuatan Ringkasan Transparan (*Transparent Summary .txt*)
Berdasarkan checklist penjaminan mutu dan integritas data, disusun laporan ringkasan ringkas dan transparan dalam berkas `.txt` yang mencatat metrik pipeline, status observasi Street View, dan penegasan anti-fabrikasi.

In [7]:
summary_notebook_path = REPO_ROOT / "notebooks" / "analisis_summary.txt"
summary_reports_path = REPO_ROOT / "reports" / "analisis_summary.txt"

summary_text = f"""RINGKASAN TRANSPARAN ANALISIS SPASIAL & DATASET BUKA KAFE SURABAYA
================================================================================
Mata Kuliah : Computer Graphics (AFL 2) — Universitas Ciputra Surabaya
Tanggal     : 2026-11-01
Komitmen    : Anti-Fabrikasi, Standar Epistemik Faktual, Koordinasi Kontrak Backend

1. STATUS KANDIDAT LAHAN (SURVEI OBSERVASI STREET VIEW NYATA - PR#2)
--------------------------------------------------------------------------------
- Sumber Data           : Google Street View (Observasi Tim & Verifikasi Metadata Pano)
- Total Titik Observasi : {len(features)} titik
- Titik Dalam Buffer 1km: {stats['in_buffer_count']} titik (100.0%)
- Titik Memenuhi Syarat : {stats['eligible_count']} titik (Lahan kosong / Tersedia disewakan)
- Titik Tidak Memenuhi  : {stats['ineligible_count']} titik (Rumah tinggal / Usaha lain / Belum terverifikasi)

Rincian Status Lahan Faktual:
- Lahan kosong                              : {stats['status_counts'].get('Lahan kosong', 0)} titik
- Lahan yang tersedia untuk disewakan       : {stats['status_counts'].get('Lahan yang tersedia untuk disewakan', 0)} titik
- Rumah atau bangunan tempat tinggal        : {stats['status_counts'].get('Rumah atau bangunan tempat tinggal', 0)} titik
- Bangunan yang telah digunakan fungsi lain : {stats['status_counts'].get('Bangunan yang telah digunakan untuk fungsi lain', 0)} titik
- Lahan yang statusnya belum terverifikasi  : {stats['status_counts'].get('Lahan yang statusnya belum terverifikasi', 0)} titik

2. REKAPITULASI 4 KANDIDAT YANG MEMENUHI SYARAT (ELIGIBLE UNTUK PERENCANA KAFE)
--------------------------------------------------------------------------------
1. SV-UNAIR-C-02 (Lahan berpagar disewakan Jl. Mulyorejo Utara 221)
   Jarak Kampus : 465.8 m ke Unair Kampus C | Pano ID: axuDfMvoxSFQUquRPNTw9A
2. SV-UKWMS-PAKUWON-01 (Lahan terbuka bertembok barat UKWMS Pakuwon City)
   Jarak Kampus : 258.6 m ke UKWMS Pakuwon | Pano ID: TVaYBWVuh8laZfloweA2ag
3. SV-UKWMS-PAKUWON-02 (Lahan terbuka berumput barat laut UKWMS Pakuwon City)
   Jarak Kampus : 474.4 m ke UKWMS Pakuwon | Pano ID: mBCxYcHyQBRJri7Oz0I4BQ
4. SV-UBAYA-TENGGILIS-01 (Lahan berpagar ditumbuhi semak Jl. Raya Kalirungkut)
   Jarak Kampus : 238.7 m ke Ubaya Tenggilis | Pano ID: LtHbBYRcFXzeHL-Nn7AfwA

3. STATUS BUKTI HARGA SEWA (RENTAL PRICE REALITY CHECK)
--------------------------------------------------------------------------------
- Observasi Street View : Luas tanah dan harga sewa = NULL (papan iklan tidak memuat angka)
- Bukti Listing Publik  : 15 listing terverifikasi langsung via web portal (Bagian 1 research.md)
- Kuorum Rentang Pasar  :
  * 7 Koridor (< 3 listing)  : DECLARED UNAVAILABLE (anti-fabrikasi)
  * 1 Koridor (>= 3 listing) : Ubaya Tenggilis (Rentang Rp 65 Jt - Rp 123,5 Jt; Median Rp 70 Jt)

4. AUDIT AKSES PANORAMA HEADLESS / BROWSER AUTOMATION
--------------------------------------------------------------------------------
- Percobaan Safaridriver    : Ditolak (Memerlukan otorisasi Remote Automation di Safari)
- Percobaan AppleScript     : Ditolak (Memerlukan izin JavaScript from Apple Events)
- Percobaan Headless Chrome : Tidak terpasang di sistem operasi
- Percobaan API Static/Pano : Ditolak (HTTP 403 / REQUEST_DENIED karena membutuhkan Google Maps API Key berbayar)
- Kesimpulan Faktual        : Tidak ada panorama baru yang diklaim telah dilihat secara interaktif oleh agen.
                              Data observasi tetap menggunakan 16 titik riil PR#2 tanpa fabrikasi.
================================================================================
"""

for sp in [summary_notebook_path, summary_reports_path]:
    sp.parent.mkdir(parents=True, exist_ok=True)
    with open(sp, "w", encoding="utf-8") as f:
        f.write(summary_text)

print(f"✓ Berhasil menulis ringkasan transparan ke:")
print(f"  1. {summary_notebook_path}")
print(f"  2. {summary_reports_path}")
print("\n--- ISI RINGKASAN ---")
print(summary_text[:1200] + "\n[... truncated preview ...]")

✓ Berhasil menulis ringkasan transparan ke:
  1. /Users/josehagen/Documents/GitHub/Comgraph-AFL-2/notebooks/analisis_summary.txt
  2. /Users/josehagen/Documents/GitHub/Comgraph-AFL-2/reports/analisis_summary.txt

--- ISI RINGKASAN ---
RINGKASAN TRANSPARAN ANALISIS SPASIAL & DATASET BUKA KAFE SURABAYA
Mata Kuliah : Computer Graphics (AFL 2) — Universitas Ciputra Surabaya
Tanggal     : 2026-11-01
Komitmen    : Anti-Fabrikasi, Standar Epistemik Faktual, Koordinasi Kontrak Backend

1. STATUS KANDIDAT LAHAN (SURVEI OBSERVASI STREET VIEW NYATA - PR#2)
--------------------------------------------------------------------------------
- Sumber Data           : Google Street View (Observasi Tim & Verifikasi Metadata Pano)
- Total Titik Observasi : 16 titik
- Titik Dalam Buffer 1km: 16 titik (100.0%)
- Titik Memenuhi Syarat : 4 titik (Lahan kosong / Tersedia disewakan)
- Titik Tidak Memenuhi  : 12 titik (Rumah tinggal / Usaha lain / Belum terverifikasi)

Rincian Status Lahan Faktual:
- Lahan koson

## 8. Kesimpulan & Penegakan Integritas Ilmiah
1. **Koordinasi Kontrak Sempurna:** Modul analisis ini mengimpor langsung logika dari `scripts.gis_pipeline`, menjamin 100% konsistensi dengan backend pipeline produksi.
2. **Kepatuhan Anti-Fabrikasi:** Kandidat lahan yang digunakan adalah 16 titik nyata hasil observasi Google Street View (PR#2). Tidak ada titik survei gerbang (*campus gates*) atau jalan raya rekaan yang ditambahkan tanpa bukti keterlihatan visual langsung.
3. **Pemisahan Data Faktual vs Simulasi:** Lahan observasi mempertahankan nilai sewa `None` (jujur), sementara penentuan rentang harga pasar dibatasi oleh aturan kuorum $\ge 3$ listing.